#### 1. Librerías.

In [8]:
%run "./librerias/librerias.ipynb"

#### 2. Constantes.

In [9]:
#a. Modo de ejecución (se define en ./constantes/modo.txt, un solo lugar para los 4 notebooks).
# "validacion" = entreno solo con train, puedo medir nDCG.
# "entrega"    = entreno con train+test, uso todo el historial para predecir.
with open("./constantes/modo.txt") as f:
    MODO = f.read().strip()

assert MODO in ("validacion", "entrega"), f"MODO inválido: {MODO!r}"
print(f"MODO: {MODO}")

MODO: entrega


In [10]:
#b. Otras constantes.
%run "./constantes/constantes.ipynb"

In [11]:
#c. Verificación del modo (que los paths coincidan con lo que creo que estoy corriendo).
print(f"MODO: {MODO} | sufijo: {sufijo!r}")
print(f"train_fe: {path_train_fe}")
print(f"modelo:   {path_modelo}")

MODO: entrega | sufijo: '_entrega'
train_fe: ./inputs/train_fe_entrega.csv
modelo:   ./modelos/modelo_entrega.pkl


#### 3. Funciones.

In [12]:
%run "./funciones/funciones.ipynb"

#### 4. Lectura.

In [13]:
#a. Conexión a la BBDD.
conn = sqlite3.connect(path_db)

In [14]:
#b. Leo las tablas de la BBDD como dataframes de pandas.
#i. Veo que tablas hay en la BBDD.
print("Tablas existentes:")
tablas = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(tablas)
#ii. Leo las tablas.
print("\nLeyendo las tablas...")
try:
    df_libros = pd.read_sql_query("SELECT * FROM libros;", conn)
    df_lectores = pd.read_sql_query("SELECT * FROM lectores;", conn)
    df_interacciones = pd.read_sql_query("SELECT * FROM interacciones;", conn)
except Exception as e:
    print(f"Error al leer las tablas: {e}")
else:
    print("Tablas leídas correctamente.")
#iii. Cierro la conexión a la BBDD.
conn.close()

Tablas existentes:
            name
0         libros
1       lectores
2  interacciones

Leyendo las tablas...
Tablas leídas correctamente.


In [15]:
#c. Leo df_train y df_test, y armo la base según el modo.
df_train_crudo = pd.read_csv(path_train_crudo, dtype={"id_lector": str, "id_libro": str})

if MODO == "validacion":
    df_train = df_train_crudo
else:
    df_test_crudo = pd.read_csv(path_test_crudo, dtype={"id_lector": str, "id_libro": str})
    df_train = pd.concat([df_train_crudo, df_test_crudo], ignore_index=True)

print(f"Modo: {MODO} | Filas de la base: {len(df_train)}")

Modo: entrega | Filas de la base: 461407


#### 5. Limpieza básica.

In [16]:
#a. Libros.
#i. Corrijo el Año de edición.
df_libros = limpiar_anio_edicion(df_libros,anio_actual)
#ii. Normalizo las columnas de tipo texto ---> Paso los valores a minúsculas y saco tildes.
for col in ["titulo", "autor", "genero", "editorial", "resumen"]:
    df_libros[col] = df_libros[col].str.lower().apply(quitar_tildes)
#iii. Lleno los vacíos de "resumen" con guiones (placeholder explícito, no NaN silencioso).
df_libros["resumen"] = df_libros["resumen"].fillna("-")
#iv. Renombro genero.
df_libros.rename(
    {
        "genero":"genero_libro"
    },axis=1,inplace=True
)

Nulos tras extraer año válido: 78305
Años fuera de rango [1800, 2024] nulificados: 717
Nulos tras imputar por promedio de título: 78969
Nulos tras imputar por promedio de editorial: 78275
Nulos tras imputar por mediana general: 0


In [17]:
#b. Lectores.
#i. Corrijo el año de nacimiento.
df_lectores = limpiar_nacimiento(df_lectores,anio_actual)
#ii.  Normalizo las columnas de tipo texto ---> Paso los valores a minúsculas y saco tildes.
for col in ["nombre", "genero","vive_en"]:
    df_lectores[col] = df_lectores[col].str.lower().apply(quitar_tildes)
#iii. Renombro genero.
df_lectores.rename(
    {
        "genero":"genero_persona"
    },axis=1,inplace=True
)

Nulos tras forzar a numérico: 3429
Nacimientos fuera de rango [1900, 2024] nulificados: 0
Nulos tras imputar por promedio de nombre: 3167
Nulos tras imputar por promedio global: 0
Nulos restantes: 0


In [18]:
#c. Interacciones.
#i. Corrijo Fecha.
#1. Train.
# Convierto a datetime, lo que no matchee queda NaT.
df_train["fecha"] = pd.to_datetime(df_train["fecha"], format="%d-%m-%Y", errors="coerce")
# Saco las filas con fecha no parseable (no tiene sentido imputar una fecha de interacción).
df_train = df_train.dropna(subset=["fecha"])

#2. Test.
# Convierto a datetime, lo que no matchee queda NaT.
#df_test["fecha"] = pd.to_datetime(df_test["fecha"], format="%d-%m-%Y", errors="coerce")
# Saco las filas con fecha no parseable (no tiene sentido imputar una fecha de interacción).
#df_test = df_test.dropna(subset=["fecha"])

#### 6. Agrupamiento de columnas.

In [19]:
#a. Libros.
#i. Agrupo genero_libro en categorías más amplias, pensando en un sistema de recomendación.
df_libros["genero_libro_agrupado"] = df_libros["genero_libro"].apply(mapear_genero_libro)
#2. Agrupo editorial: top-N + "otras", conservando frecuencia para la cola larga (top 30, que cubre bastante más del 50% ya).
top_n_editoriales = df_libros["editorial"].value_counts().head(30).index.tolist()

df_libros["editorial_agrupada"] = np.where(df_libros["editorial"].isin(top_n_editoriales),
                                           df_libros["editorial"],
                                           "otras"
                                           )

In [20]:
#b. Lectores.
#i. Vive_en: separo en pais y ciudad, limpiando placeholders.
#1. Separo por " - ": si hay 2 partes, la primera es ciudad y la segunda país.
vive_en_str = df_lectores["vive_en"].astype(str).str.strip()
vive_en_str = vive_en_str.replace("nan", np.nan)

#2. Caso especial: strings que terminan en "-" (el país quedó vacío tras el strip, ej. "Madrid -").
termina_en_guion = vive_en_str.str.endswith("-", na=False)
vive_en_str = vive_en_str.where(~termina_en_guion, vive_en_str + " ")

partes = vive_en_str.str.split(" - ", n=1, expand=True)

ciudad_raw = np.where(partes[1].notna(), partes[0].str.strip(), np.nan)
pais_raw = np.where(partes[1].notna(), partes[1].str.strip(), partes[0].str.strip())

df_lectores["ciudad"] = pd.Series(ciudad_raw, index=df_lectores.index)
df_lectores["pais"] = pd.Series(pais_raw, index=df_lectores.index)

#3. Limpio placeholders (vacíos, variantes de "¿?") en CADA columna resultante.
patron_placeholder = r"^\s*$|^\W*\?+\W*$"
df_lectores["ciudad"] = df_lectores["ciudad"].replace(patron_placeholder, np.nan, regex=True)
df_lectores["pais"] = df_lectores["pais"].replace(patron_placeholder, np.nan, regex=True)


#4. Infiero pais faltante a partir de ciudad, usando el país más frecuente para esa ciudad (cuando ambos son conocidos).
mapa_ciudad_pais = (
    df_lectores.dropna(subset=["ciudad", "pais"])
    .groupby("ciudad")["pais"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

mask_pais_faltante = df_lectores["pais"].isna() & df_lectores["ciudad"].notna()

pais_inferido = df_lectores.loc[mask_pais_faltante, "ciudad"].map(mapa_ciudad_pais)
df_lectores.loc[mask_pais_faltante, "pais"] = pais_inferido

#5. Agrupo pais: top-N + "otras", igual que hicimos con editorial.
top_n_paises = df_lectores["pais"].value_counts().head(20).index.tolist()
df_lectores["pais_agrupado"] = np.where(
    df_lectores["pais"].isin(top_n_paises),
    df_lectores["pais"],
    "otras"
)


#### 7. Creación de nuevas columnas.

In [21]:
#a. Mergeo las tablas.
#i. Train.
df_total_train = df_train.merge(df_libros,how="left",on="id_libro").merge(df_lectores,how="left",on="id_lector")
#ii. Test.
#df_total_test = df_test.merge(df_libros,how="left",on="id_libro").merge(df_lectores,how="left",on="id_lector")

In [22]:
#b. Calculo edad del lector al momento de la interacción (año de la interacción, no de la edición del libro).
#i. Train.
df_total_train["edad_al_interactuar"] = df_total_train["fecha"].dt.year - df_total_train["nacimiento"]
#ii. Test.
#df_total_test["edad_al_interactuar"] = df_total_test["fecha"].dt.year - df_total_test["nacimiento"]

In [23]:
#c. Calculo cantidad de dias que pasaron de la interacción, tomando como referencia el 31/12/2024.
#################################################################################################
#################################################################################################
# LO SACO PORQUE EN TEST Y EN A PREDECIR VA A SER TODO 0. NO ME SIRVE COMO VARIABLE INFORMATIVA.
#################################################################################################
#################################################################################################
#fecha_referencia = pd.Timestamp(year=anio_actual, month=12, day=31)
#i. Train.
#df_total_train["dias_transcurridos_interaccion"] = (fecha_referencia - df_total_train["fecha"]).dt.days
#i. Test.
#df_total_test["dias_transcurridos_interaccion"] = (fecha_referencia - df_total_test["fecha"]).dt.days

In [24]:
#d. Calculo años transcurridos desde la edición.
#1. Train.
#i. Hasta la interacción.
df_total_train["anios_transcurridos_edicion"] = df_total_train["fecha"].dt.year - df_total_train["anio_edicion"]
#ii. Hasta hoy.
########################################################################################################################################
################################## Lo saco porque en test va a ser igual a anios_transcurridos_edicio ##################################
########################################################################################################################################
#df_total_train["antiguedad_libro_hoy"] = anio_actual - df_total_train["anio_edicion"]

#2. Test.
#i. Hasta la interacción.
#df_total_test["anios_transcurridos_edicion"] = df_total_test["fecha"].dt.year - df_total_test["anio_edicion"]
#ii. Hasta hoy.
#df_total_test["antiguedad_libro_hoy"] = anio_actual - df_total_test["anio_edicion"]

In [25]:
#e. Frecuencia de lectura (leave-one-out: no cuento el registro actual).
#1. Creación de tablas de referencia.
#i. Frecuencia Lector.
freq_lector = (df_total_train.groupby("id_lector").size().rename("freq_lector_train"))
#ii. Frecuencia Libro.
freq_libro = (df_total_train.groupby("id_libro").size().rename("freq_libro_train"))
#iii. Lector x Autor.
freq_lector_autor = (df_total_train.groupby(["id_lector", "autor"]).size().rename("freq_la_train"))
#iv. Lector x Género.
freq_lector_genero = (df_total_train.groupby(["id_lector", "genero_libro_agrupado"]).size().rename("freq_lg_train"))
#v. Popularidad de Autor (Lectores distintos).
pop_autor = (df_total_train.groupby("autor")["id_lector"].nunique().rename("pop_autor_train"))

#2. Train.
#i. Lector.
df_total_train["frecuencia_lector"] = df_total_train["id_lector"].map(freq_lector) - 1
#ii. Libro. 
df_total_train["frecuencia_libro"] = df_total_train["id_libro"].map(freq_libro) - 1
#iii. Lector x Autor (cuántas veces este lector interactuó con este autor).
df_total_train["n_interacciones_lector_autor"] = (df_total_train.set_index(["id_lector", "autor"]).index.map(freq_lector_autor) - 1)
#iv. Lector x Género (idem, para género).
df_total_train["n_interacciones_lector_genero"] = (df_total_train.set_index(["id_lector", "genero_libro_agrupado"]).index.map(freq_lector_genero) - 1)
#v. Popularidad de autor: cuántos lectores distintos tuvo.
df_total_train["n_lectores_distintos_autor"] = df_total_train["autor"].map(pop_autor) - 1


#2. Test.
#i. Lector.
#df_total_test["frecuencia_lector"] = (
#    df_total_test["id_lector"].map(freq_lector).fillna(0)
#)
#ii. Libro. 
#df_total_test["frecuencia_libro"] = (
#    df_total_test["id_libro"].map(freq_libro).fillna(0)
#)
#iii. Lector x Autor (cuántas veces este lector interactuó con este autor).
#df_total_test["n_interacciones_lector_autor"] = (
#    df_total_test.set_index(["id_lector", "autor"])
#    .index.map(freq_lector_autor)
#    .fillna(0)
#)
#iv. Lector x Género (idem, para género).
#df_total_test["n_interacciones_lector_genero"] = (
#    df_total_test.set_index(["id_lector", "genero_libro_agrupado"])
#    .index.map(freq_lector_genero)
#    .fillna(0)
#)
#v. Popularidad de autor: cuántos lectores distintos tuvo.
#df_total_test["n_lectores_distintos_autor"] = (
#    df_total_test["autor"].map(pop_autor).fillna(0)
#)

In [26]:
#f. Features de rating promedio (leave-one-out, evita leakage).
#i. Media global (la necesito antes, para el shrinkage).
media_global = df_total_train["rating"].mean()

#ii. ¿Cómo suele rankear el lector? (sin shrinkage: son features de lector, no de ítem)
df_total_train["rating_prom_id_lector"] = loo_mean(df_total_train, "id_lector")
df_total_train["rating_prom_id_lector_autor"] = loo_mean(df_total_train, ["id_lector","autor"])
df_total_train["rating_prom_id_lector_genero_libro_agrupado"] = loo_mean(df_total_train, ["id_lector","genero_libro_agrupado"])

#iii. ¿Cuál suele ser el ranking global? (con shrinkage en libro y autor: son los grupos
# con cola larga, donde un libro con 1 lectura y rating 10 no puede valer lo mismo
# que un clásico con 500 lecturas y 8.7).
df_total_train["rating_prom_id_libro"] = loo_mean_shrunk(df_total_train, "id_libro", media_global, M_SHRINK)
df_total_train["rating_prom_autor"] = loo_mean_shrunk(df_total_train, "autor", media_global, M_SHRINK)
df_total_train["rating_prom_genero"] = loo_mean(df_total_train, "genero_libro_agrupado")

#iv. Fallback jerárquico para los NaN que deja el LOO cuando n=1 en el grupo.
#1. Niveles base con la media global.
# Si M_SHRINK = 0, loo_mean_shrunk vuelve a poder dar NaN (denominador n_loo),
# así que dejo id_libro y autor en la lista por las dudas.
df_total_train.fillna({
    "rating_prom_id_lector": media_global,
    "rating_prom_id_libro": media_global,
    "rating_prom_autor": media_global,
    "rating_prom_genero": media_global,
}, inplace=True)

#2. Recién ahora uso rating_prom_id_lector (ya sin NaN) como fallback para los niveles de a pares.
for col, fallback in [
    ("rating_prom_id_lector_autor", "rating_prom_id_lector"),
    ("rating_prom_id_lector_genero_libro_agrupado", "rating_prom_id_lector"),
]:
    df_total_train[col] = df_total_train[col].fillna(df_total_train[fallback])

In [27]:
#g. Diversidad del lector ---> cuántos autores/géneros distintos leyó.
#i. Tablas de referencias.
# Conteo global de únicos en Train por lector
autores_por_lector = df_total_train.groupby("id_lector")["autor"].nunique()
generos_por_lector = df_total_train.groupby("id_lector")["genero_libro_agrupado"].nunique()
# Conteo de veces que el lector leyó a ESTE autor / género en Train
veces_lector_autor = df_total_train.groupby(["id_lector", "autor"]).size()
veces_lector_genero = df_total_train.groupby(["id_lector", "genero_libro_agrupado"]).size()

#ii. Train.
# Si el lector leyó a este autor EXACTAMENTE 1 VEZ en Train,  al quitar esta fila pierde 1 autor distinto. 
# Si lo leyó >1 vez, no pierde el autor.
es_unica_lectura_autor = (df_total_train.set_index(["id_lector", "autor"]).index.map(veces_lector_autor) == 1).astype(int)
df_total_train["n_autores_distintos_lector"] = (df_total_train["id_lector"].map(autores_por_lector) - es_unica_lectura_autor)
# Misma lógica para Género
es_unica_lectura_genero = (df_total_train.set_index(["id_lector", "genero_libro_agrupado"]).index.map(veces_lector_genero) == 1).astype(int)
df_total_train["n_generos_distintos_lector"] = (df_total_train["id_lector"].map(generos_por_lector) - es_unica_lectura_genero)

#iii. Test.
#df_total_test["n_autores_distintos_lector"] = (df_total_test["id_lector"].map(autores_por_lector).fillna(0))
#df_total_test["n_generos_distintos_lector"] = (df_total_test["id_lector"].map(generos_por_lector).fillna(0))

In [28]:
#h. Proporciones de afinidad (leave-one-out: numerador y denominador ya vienen con -1).
# Para un lector con 500 libros, "leí 5 de este autor" no discrimina entre candidatos;
# "el 1% de mi historial es de este autor" sí. Ataca el bucket de lectores pesados,
# que es donde el nDCG cae (0.155 en 1-20 vs 0.045 en 250+).
#i. Denominador seguro: un lector con 1 sola interacción queda en 0 tras el LOO.
den_lector = df_total_train["frecuencia_lector"].replace(0, np.nan)

#ii. Qué proporción del historial del lector es de este autor / este género.
df_total_train["prop_lector_autor"] = df_total_train["n_interacciones_lector_autor"] / den_lector
df_total_train["prop_lector_genero"] = df_total_train["n_interacciones_lector_genero"] / den_lector

#iii. Qué tan explorador es el lector (constante por lector: no ordena solo, pero le da
# contexto al modelo para calibrar cuánto pesar las dos de arriba. Un lector con 200 libros
# y 190 autores distintos usa la afinidad por autor muy distinto de uno con 200 y 20).
df_total_train["prop_autores_distintos"] = df_total_train["n_autores_distintos_lector"] / den_lector
df_total_train["prop_generos_distintos"] = df_total_train["n_generos_distintos_lector"] / den_lector

#iv. Fallback: lector con una sola interacción -> sin historial previo, proporción 0.
df_total_train[["prop_lector_autor", "prop_lector_genero",
                "prop_autores_distintos", "prop_generos_distintos"]] = (
    df_total_train[["prop_lector_autor", "prop_lector_genero",
                    "prop_autores_distintos", "prop_generos_distintos"]].fillna(0)
)

In [29]:
#i. Perfil de popularidad del lector.
# La idea: no importa solo si el libro es popular, sino si es MÁS popular de lo que
# este lector suele leer. Un lector de nicho y uno de best-sellers ordenan distinto.
# log1p porque frecuencia_libro tiene cola larguísima y la diferencia en crudo la domina.
#i. Popularidad del libro en escala log.
df_total_train["log_pop_libro"] = np.log1p(df_total_train["frecuencia_libro"])

#ii. Popularidad media de los libros que leyó el lector (LOO: excluyo la fila actual,
# si no le estoy filtrando al modelo que este lector leyó este libro).
grp = df_total_train.groupby("id_lector")["log_pop_libro"]
df_total_train["log_pop_media_lector"] = (
    (grp.transform("sum") - df_total_train["log_pop_libro"]) / (grp.transform("count") - 1)
)
df_total_train["log_pop_media_lector"] = df_total_train["log_pop_media_lector"].fillna(
    df_total_train["log_pop_libro"].mean()
)

#iii. La que ordena: cuánto se desvía este candidato del perfil del lector.
df_total_train["dif_log_pop"] = (
    df_total_train["log_pop_libro"] - df_total_train["log_pop_media_lector"]
)

In [30]:
#j. Codifico categóricas de baja/media cardinalidad con dummies.
#i. Columnas a convertir.
cols_dummies = ["genero_persona","pais_agrupado","genero_libro_agrupado", "editorial_agrupada"]
#ii. Train.
# Creo las dummies SIN modificar df_total_train
df_dummies = pd.get_dummies(
    df_total_train[cols_dummies],
    columns=cols_dummies,
    drop_first=False
)

# Agrego las dummies al dataframe original (esto lo hago para luego poder mergear test).
df_total_train = pd.concat(
    [df_total_train, df_dummies],
    axis=1
)
#iii. Test.
#df_total_test = pd.get_dummies(df_total_test, columns=cols_dummies, drop_first=False)
#iv. Alineo test con las columnas exactas de train.
#df_total_train, df_total_test = df_total_train.align(df_total_test, join="left", axis=1, fill_value=0)

#### 8. Exportación.

In [31]:
#a. Train.
df_total_train.to_csv(path_train_fe,index=False)

In [32]:
#b. Test.
#df_total_test.to_csv(path_test_fe,index=False)

In [33]:
#c. Libros.
df_libros.to_csv(path_libros_fe,index=False)

In [34]:
#d. Lectores.
df_lectores.to_csv(path_lectores_fe,index=False)